# External Dataset Acquisition and Immutable Byte Freeze

## Purpose

This notebook reconstructs the complete five-dataset external evidence package for the Swarabyanjan reliability study from immutable public sources in a single execution. It has no dependency on outputs from earlier acquisition notebooks and does not require Mendeley credentials.

The operation class is **acquisition and physical-byte verification**. No preprocessing, deduplication, feature extraction, modeling, calibration, statistical analysis, or evaluation is performed.

## Dataset provenance

The immutable source identities are:

- **BanglaBait** — author Hugging Face dataset `Motahar/banglabait`, revision `ff22fb3b423952959a80dc5d9c6c3345b21cd145`.
- **BanglaClick** — author GitHub repository `MBillahsust/BanglaClick`, commit `2ba28763d8f80eaba7a649877eae1fbbcb78a0e7`.
- **BanMANI** — author GitHub repository `kamruzzaman15/BanMANI`, commit `483be9ac31f550ee252d1df99d090ffdbcccc63f`.
- **BanFakeNews-2.0** — authoritative release identity Mendeley Data Version 1, DOI `10.17632/kjh887ct4j.1`; public transport is accepted only when the acquired physical bytes match the frozen Version 1 target.
- **BaitBuster-Bangla** — authoritative release identity Mendeley Data Version 3, DOI `10.17632/3c6ztw5nft.3`. The author-owned pinned Version 3 Parquet artifact is the canonical research serialization for downstream P0 analysis and experiments. The author-owned Version 3 XLSX artifact is retained as a secondary provenance/reference serialization. The publisher-listed Version 3 CSV serialization is documented as unavailable through the verified public sources and is not required for the scientific freeze.

The BaitBuster Parquet artifact is not described as byte-identical to the CSV serialization. It is independently verified against the locked Version 3 Parquet byte target.

## Input requirements

No previous acquisition directory is required.

Required runtime inputs are limited to:

- Internet access to Hugging Face, GitHub, and the public Kaggle dataset API where used by BanFakeNews-2.0.
- Writable storage under `/kaggle/working` when executed on Kaggle.
- Sufficient local storage for source files and the final immutable ZIP.

No Kaggle `Add Input` dataset attachment is required for the five target datasets.

## Software dependencies and runtime assumptions

- Python 3.10 or later.
- `requests`, `huggingface_hub`, `tqdm`, and Python standard-library archive/hash modules.
- CPU execution is sufficient. GPU access is not required or used.
- A minimum of 6 GiB of free working storage is enforced before acquisition.

## Network requirements

Internet access is required. The notebook performs no request to a Mendeley metadata or file-delivery endpoint. Mendeley URLs are retained only as authoritative release identifiers in provenance records.

## Output artifacts

A complete successful execution creates:

- `EXTERNAL-DATASET-BYTES-v1/SOURCES_LOCK.json`
- `EXTERNAL-DATASET-BYTES-v1/ACQUISITION_STATUS.json`
- `EXTERNAL-DATASET-BYTES-v1/ACQUISITION_MANIFEST.json`
- `EXTERNAL-DATASET-BYTES-v1/SHA256SUMS.txt`
- `EXTERNAL-DATASET-BYTES-v1/README_HANDOFF.txt`
- `EXTERNAL-DATASET-BYTES-v1.zip`
- `EXTERNAL-DATASET-BYTES-v1.zip.sha256`

Dataset-specific `SOURCE_METADATA.json` records are also written inside each verified dataset directory.

## Failure conditions

Final ZIP creation is prohibited when any required canonical dataset artifact fails acquisition, source-identity verification, required-file discovery, byte-size verification, cryptographic verification, or archive-integrity verification.

For BaitBuster-Bangla, the required canonical artifact is the pinned Version 3 Parquet file. XLSX is secondary and is verified and retained when available. The unavailable publisher-listed CSV is not a freeze requirement and is never reconstructed.

## Reproducibility and license notes

Source revision/commit, retrieval route, required filename, physical byte size, cryptographic identity, serialization role, and retrieval timestamp are recorded. Source licenses remain those assigned by the original publishers/repositories. A transport copy does not change the license or authoritative release identity of the underlying dataset.


In [1]:

%pip -q install "requests>=2.31,<3" "huggingface_hub>=0.27,<2" "tqdm>=4.66,<5"


Note: you may need to restart the kernel to use updated packages.


## Frozen source registry and runtime configuration

In [2]:

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import os
import platform
import shutil
import sys
import tarfile
import time
import urllib.parse
import zipfile

import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from huggingface_hub import HfApi, snapshot_download, hf_hub_download
from tqdm.auto import tqdm

PACKAGE_NAME = "EXTERNAL-DATASET-BYTES-v1"
OUTPUT_PARENT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
ROOT = OUTPUT_PARENT / PACKAGE_NAME
STAGING = OUTPUT_PARENT / f"{PACKAGE_NAME}__staging"
ZIP_PATH = OUTPUT_PARENT / f"{PACKAGE_NAME}.zip"
ZIP_SHA_PATH = OUTPUT_PARENT / f"{PACKAGE_NAME}.zip.sha256"

MIN_FREE_BYTES = 6 * 1024**3
REQUIRED_DATASETS = [
    "BanglaBait",
    "BanglaClick",
    "BanMANI",
    "BanFakeNews-2.0",
    "BaitBuster-Bangla",
]

SOURCES = {
    "BanglaBait": {
        "kind": "huggingface_snapshot",
        "repo_id": "Motahar/banglabait",
        "repo_type": "dataset",
        "revision": "ff22fb3b423952959a80dc5d9c6c3345b21cd145",
        "source_url": "https://huggingface.co/datasets/Motahar/banglabait",
        "license_note": "Repository metadata identifies CC BY-NC 4.0.",
        "required_files": ["train.csv", "dev.csv", "test.csv", "unlabelled_data.csv"],
        "expected_tree": {
            ".gitattributes": {
                "size": 2608,
                "git_blob_sha1": "f0632c67bb7aa0706fca4150b4d813b0e3f983e5",
            },
            "README.md": {
                "size": 111,
                "git_blob_sha1": "9ba86c6c3de662c669383b0e0a3432678feb1fa5",
            },
            "dev.csv": {
                "size": 7969172,
                "git_blob_sha1": "835fcc6576a33bcc66c400091383998ffd059680",
            },
            "test.csv": {
                "size": 19813408,
                "sha256": "68176be61ba8dd01f9f78b6aea2b84c198236c89de5abc98db8c49c496ebd996",
            },
            "train.csv": {
                "size": 72704138,
                "sha256": "97f4be47da54821904d57028b53d539dcb3cbf76942cf3ac530a8c313eec06f5",
            },
            "unlabelled_data.csv": {
                "size": 329277343,
                "sha256": "66c6186e3e351c7703cc5d8a7061957e708e8c55ccaea9bf24f55875a7fed583",
            },
        },
    },
    "BanglaClick": {
        "kind": "github_commit",
        "owner": "MBillahsust",
        "repo": "BanglaClick",
        "source_url": "https://github.com/MBillahsust/BanglaClick",
        "repo_url": "https://github.com/MBillahsust/BanglaClick.git",
        "commit": "2ba28763d8f80eaba7a649877eae1fbbcb78a0e7",
        "required_files": [
            "Data/Final_4961_Dataset.xlsx",
            "README.md",
            "Data/README.md",
        ],
        "sha256_policy": (
            "No independent pre-frozen SHA-256 was defined for these Git objects. "
            "The pinned commit and Git blob SHA-1 establish the expected bytes; "
            "SHA-256 is independently computed and frozen by this execution."
        ),
        "license_note": (
            "Repository README identifies CC BY 4.0 for annotations and MIT for code; "
            "third-party text is not redistributed by the repository."
        ),
    },
    "BanMANI": {
        "kind": "github_commit",
        "owner": "kamruzzaman15",
        "repo": "BanMANI",
        "source_url": "https://github.com/kamruzzaman15/BanMANI",
        "repo_url": "https://github.com/kamruzzaman15/BanMANI.git",
        "commit": "483be9ac31f550ee252d1df99d090ffdbcccc63f",
        "required_files": ["BanMANI.csv", "README.md", "LICENSE"],
        "sha256_policy": (
            "No independent pre-frozen SHA-256 was defined for these Git objects. "
            "The pinned commit and Git blob SHA-1 establish the expected bytes; "
            "SHA-256 is independently computed and frozen by this execution."
        ),
        "license_note": "The repository LICENSE file is preserved verbatim in the frozen snapshot.",
    },
    "BanFakeNews-2.0": {
        "kind": "authoritative_release_with_public_transport",
        "authoritative": {
            "platform": "Mendeley Data",
            "version": 1,
            "doi": "10.17632/kjh887ct4j.1",
            "source_url": "https://data.mendeley.com/datasets/kjh887ct4j/1",
        },
        "author_kaggle": {
            "owner": "hrithikmajumdar",
            "slug": "bangla-fake-news",
            "source_url": "https://www.kaggle.com/datasets/hrithikmajumdar/bangla-fake-news",
        },
        "public_hf_candidate": {
            "repo_id": "hrshihab/BanFakeNews-2.0",
            "revision": "cef06ca5e2a90c408a28719d6d71c1e1388c3f7a",
            "source_url": "https://huggingface.co/datasets/hrshihab/BanFakeNews-2.0",
            "provenance_note": (
                "Public transport candidate rather than authoritative publisher. "
                "Its pinned stored-object pointer equals the frozen authoritative target."
            ),
        },
        "files": {
            "All60Kdataset.csv": {
                "size": 304586219,
                "sha256": "d9aed69b6019843a769bbcb045affd31c98cfa02f2b0bc821f551c54ba780c0a",
            },
        },
        "new_test": {
            "repo_url": "https://github.com/Shibu4064/IndoNLP.git",
            "source_url": "https://github.com/Shibu4064/IndoNLP",
            "commit": "8736467eeea8a441a35508777fcf6c8cdd323df7",
            "filename": "New_Test_Dataset.csv",
            "size": 6237141,
            "sha256": "ed4949762c1cdc1940bb5a1afe610ca1341a0803079888a35a812608f6351e7b",
            "raw_url": (
                "https://raw.githubusercontent.com/Shibu4064/IndoNLP/"
                "8736467eeea8a441a35508777fcf6c8cdd323df7/New_Test_Dataset.csv"
            ),
        },
    },
    "BaitBuster-Bangla": {
        "kind": "authoritative_release_with_canonical_public_serialization",
        "authoritative": {
            "platform": "Mendeley Data",
            "version": 3,
            "doi": "10.17632/3c6ztw5nft.3",
            "source_url": "https://data.mendeley.com/datasets/3c6ztw5nft/3",
        },
        "author_github": {
            "source_url": "https://github.com/abdalimran/BaitBuster-Bangla",
        },
        "author_hf": {
            "repo_id": "abdalimran/BaitBuster-Bangla",
            "revision": "be1528ba40dd8828c5c06ee70b7574cca22b8ee1",
            "source_url": "https://huggingface.co/datasets/abdalimran/BaitBuster-Bangla",
        },
        "canonical_research_artifact": {
            "filename": "BaitBuster-Bangla_253070_18c_HL10k_AIL.parquet",
            "serialization": "parquet",
            "size": 175950911,
            "sha256": "7f1bb757c9c5ee1a2922efcd45fb829b89e96d4465995e538158c4225f3c12c9",
            "transport": "author_hf_pinned_commit",
            "required_for_pass": True,
        },
        "secondary_reference_artifact": {
            "filename": "BaitBuster-Bangla_253070_18c_HL10k_AIL.xlsx",
            "serialization": "xlsx",
            "size": 91845301,
            "sha256": "a6d207d94ba0e4735a92478ce01c0080f79e9ce2d1557c78c304e5275b3d5286",
            "transport": "author_hf_pinned_commit",
            "required_for_pass": False,
        },
        "publisher_listed_unavailable_serialization": {
            "filename": "BaitBuster-Bangla_253070_18c_HL10k_AIL.csv",
            "serialization": "csv",
            "size": 936567497,
            "sha256": "c94e1abd737fa53f983de2e3de2fc322757f5548024e463d8365c0e2a0e4ce63",
            "public_recovery_status": "unavailable_in_verified_public_sources",
            "required_for_pass": False,
            "reconstruction_prohibited": True,
        },
        "license_note": (
            "Original publisher/repository license terms remain controlling."
        ),
    },
}

print("Python:", sys.version.replace("\n", " "))
print("Platform:", platform.platform())
print("Output root:", ROOT)
print("Staging root:", STAGING)
print("Free bytes:", f"{shutil.disk_usage(OUTPUT_PARENT).free:,}")

if shutil.disk_usage(OUTPUT_PARENT).free < MIN_FREE_BYTES:
    raise RuntimeError(
        f"Insufficient free storage: {shutil.disk_usage(OUTPUT_PARENT).free:,} bytes; "
        f"minimum required is {MIN_FREE_BYTES:,} bytes."
    )


Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Platform: Linux-6.12.90+-x86_64-with-glibc2.35
Output root: /kaggle/working/EXTERNAL-DATASET-BYTES-v1
Staging root: /kaggle/working/EXTERNAL-DATASET-BYTES-v1__staging
Free bytes: 20,940,423,168


## Acquisition and verification utilities

In [3]:

retry = Retry(
    total=4,
    connect=4,
    read=4,
    status=4,
    backoff_factor=1.5,
    status_forcelist=(408, 425, 429, 500, 502, 503, 504),
    allowed_methods=frozenset({"GET"}),
    respect_retry_after_header=True,
    raise_on_status=False,
)
session = requests.Session()
session.mount("https://", HTTPAdapter(max_retries=retry, pool_connections=8, pool_maxsize=8))
session.headers.update({
    "User-Agent": "Swarabyanjan-Reproducibility-Freeze/2.0",
    "Accept-Encoding": "identity",
})

def utc_now():
    return datetime.now(timezone.utc).isoformat()

def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8", newline="\n") as f:
        json.dump(value, f, indent=2, ensure_ascii=False, default=str)
        f.write("\n")

def sha256_file(path, chunk_size=8 * 1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                return h.hexdigest()
            h.update(chunk)

def git_blob_sha1(path, chunk_size=8 * 1024 * 1024):
    path = Path(path)
    h = hashlib.sha1()
    h.update(f"blob {path.stat().st_size}\0".encode("ascii"))
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                return h.hexdigest()
            h.update(chunk)

def verify_frozen_file(path, expected_size, expected_sha256):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(path)
    actual_size = path.stat().st_size
    if actual_size != int(expected_size):
        raise RuntimeError(
            f"SIZE_MISMATCH {path.name}: {actual_size} != {expected_size}"
        )
    actual_sha = sha256_file(path)
    if actual_sha.lower() != expected_sha256.lower():
        raise RuntimeError(
            f"SHA256_MISMATCH {path.name}: {actual_sha} != {expected_sha256}"
        )
    return {
        "path": str(path.resolve()),
        "size_bytes": actual_size,
        "sha256": actual_sha,
    }

def iter_files(root):
    root = Path(root)
    return sorted(
        [p for p in root.rglob("*") if p.is_file()],
        key=lambda p: p.relative_to(root).as_posix(),
    )

def safe_extract_tar(tar_path, destination):
    destination = Path(destination).resolve()
    destination.mkdir(parents=True, exist_ok=True)
    with tarfile.open(tar_path, "r:*") as tf:
        for member in tf.getmembers():
            target = (destination / member.name).resolve()
            if target != destination and destination not in target.parents:
                raise RuntimeError(f"Unsafe tar member: {member.name}")
        try:
            tf.extractall(destination, filter="data")
        except TypeError:
            tf.extractall(destination)

def safe_extract_named_zip_member(zip_path, expected_filename, destination):
    zip_path = Path(zip_path)
    destination = Path(destination)
    with zipfile.ZipFile(zip_path, "r") as zf:
        matches = [
            name for name in zf.namelist()
            if not name.endswith("/") and Path(name).name == expected_filename
        ]
        if len(matches) != 1:
            inventory = [n for n in zf.namelist() if not n.endswith("/")][:100]
            raise RuntimeError(
                f"Expected exactly one ZIP member named {expected_filename!r}; "
                f"found {matches}. First archive members: {inventory}"
            )
        destination.parent.mkdir(parents=True, exist_ok=True)
        with zf.open(matches[0], "r") as src, open(destination, "wb") as dst:
            shutil.copyfileobj(src, dst, length=8 * 1024 * 1024)

def download_stream(url, destination, timeout=(30, 900), headers=None):
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    tmp = destination.with_suffix(destination.suffix + ".part")
    tmp.unlink(missing_ok=True)

    with session.get(
        url,
        stream=True,
        timeout=timeout,
        allow_redirects=True,
        headers=headers,
    ) as r:
        if r.status_code != 200:
            excerpt = ""
            try:
                excerpt = r.text[:400].replace("\n", " ")
            except Exception:
                pass
            raise RuntimeError(f"HTTP {r.status_code} for {url}: {excerpt}")

        content_length = r.headers.get("content-length")
        total = int(content_length) if content_length and content_length.isdigit() else None

        with open(tmp, "wb") as f, tqdm(
            total=total,
            unit="B",
            unit_scale=True,
            desc=destination.name,
        ) as bar:
            for chunk in r.iter_content(8 * 1024 * 1024):
                if chunk:
                    f.write(chunk)
                    bar.update(len(chunk))

        meta = {
            "requested_url": url,
            "final_url": r.url,
            "status_code": r.status_code,
            "content_type": r.headers.get("content-type"),
            "content_length": r.headers.get("content-length"),
            "content_disposition": r.headers.get("content-disposition"),
            "etag": r.headers.get("etag"),
            "last_modified": r.headers.get("last-modified"),
        }

    tmp.replace(destination)
    return meta

def github_json(url):
    headers = {
        "Accept": "application/vnd.github+json",
        "X-GitHub-Api-Version": "2022-11-28",
    }
    token = os.environ.get("GITHUB_TOKEN") or os.environ.get("GH_TOKEN")
    if token:
        headers["Authorization"] = f"Bearer {token}"

    r = session.get(url, timeout=(20, 120), headers=headers)
    if r.status_code != 200:
        raise RuntimeError(
            f"GitHub API HTTP {r.status_code} for {url}: "
            f"{r.text[:400].replace(chr(10), ' ')}"
        )
    return r.json()

def kaggle_dataset_metadata(owner, slug):
    url = f"https://www.kaggle.com/api/v1/datasets/view/{owner}/{slug}"
    try:
        r = session.get(url, timeout=(20, 120), allow_redirects=True)
        if r.status_code == 200:
            data = r.json()
            return {
                "url": url,
                "status_code": r.status_code,
                "current_version_number": (
                    data.get("currentVersionNumber")
                    or data.get("current_version_number")
                ),
                "title": data.get("title"),
                "owner_ref": data.get("ownerRef"),
            }
        return {"url": url, "status_code": r.status_code}
    except Exception as exc:
        return {"url": url, "error": f"{type(exc).__name__}: {exc}"}

def kaggle_public_file_download(owner, slug, filename, destination, staging_dir):
    """
    Retrieve a file only from the specified public Kaggle dataset.
    Exact-file API is attempted first; whole-dataset archive from the same
    owner/slug is the fallback. No authentication credential is supplied.
    """
    destination = Path(destination)
    staging_dir = Path(staging_dir)
    staging_dir.mkdir(parents=True, exist_ok=True)

    metadata = kaggle_dataset_metadata(owner, slug)
    version = metadata.get("current_version_number")
    attempts = []

    encoded_filename = urllib.parse.quote(filename, safe="")
    direct_url = (
        f"https://www.kaggle.com/api/v1/datasets/download/"
        f"{owner}/{slug}/{encoded_filename}"
    )
    if version:
        direct_url += f"?datasetVersionNumber={version}"

    direct_blob = staging_dir / f"{filename}.kaggle_exact_response"
    try:
        response = download_stream(direct_url, direct_blob)
        if zipfile.is_zipfile(direct_blob):
            safe_extract_named_zip_member(direct_blob, filename, destination)
            container = "zip"
        else:
            destination.parent.mkdir(parents=True, exist_ok=True)
            direct_blob.replace(destination)
            container = "raw"
        return {
            "route": "kaggle_public_exact_file_api",
            "dataset": f"{owner}/{slug}",
            "dataset_page": f"https://www.kaggle.com/datasets/{owner}/{slug}",
            "dataset_version_number": version,
            "download_url": direct_url,
            "container": container,
            "response": response,
            "metadata": metadata,
        }
    except Exception as exc:
        attempts.append(f"exact-file API: {type(exc).__name__}: {exc}")
        direct_blob.unlink(missing_ok=True)
        destination.unlink(missing_ok=True)

    archive_url = (
        f"https://www.kaggle.com/api/v1/datasets/download/{owner}/{slug}"
    )
    if version:
        archive_url += f"?datasetVersionNumber={version}"

    archive_path = staging_dir / f"{slug}.zip"
    try:
        response = download_stream(archive_url, archive_path)
        if not zipfile.is_zipfile(archive_path):
            raise RuntimeError("Dataset archive response is not a ZIP file")
        safe_extract_named_zip_member(archive_path, filename, destination)
        return {
            "route": "kaggle_public_dataset_archive_api",
            "dataset": f"{owner}/{slug}",
            "dataset_page": f"https://www.kaggle.com/datasets/{owner}/{slug}",
            "dataset_version_number": version,
            "download_url": archive_url,
            "container": "zip",
            "response": response,
            "metadata": metadata,
        }
    except Exception as exc:
        attempts.append(f"dataset archive API: {type(exc).__name__}: {exc}")
        raise RuntimeError(
            f"Public Kaggle acquisition failed for {owner}/{slug}/{filename}: "
            + " | ".join(attempts)
        ) from exc

def prepare_dataset_dir(name):
    stage = STAGING / name
    promoted = ROOT / name
    if stage.exists():
        shutil.rmtree(stage)
    if promoted.exists():
        shutil.rmtree(promoted)
    stage.mkdir(parents=True, exist_ok=True)
    return stage, promoted

def promote_dataset(stage, promoted):
    shutil.copytree(stage / "promoted", promoted)
    return promoted

# Clean stale final outputs while preserving no prior-run dependency.
if ROOT.exists():
    shutil.rmtree(ROOT)
if STAGING.exists():
    shutil.rmtree(STAGING)
ROOT.mkdir(parents=True, exist_ok=True)
STAGING.mkdir(parents=True, exist_ok=True)
ZIP_PATH.unlink(missing_ok=True)
ZIP_SHA_PATH.unlink(missing_ok=True)

STATUSES = {
    name: {
        "status": "NOT_RUN",
        "started_utc": None,
        "finished_utc": None,
        "detail": None,
    }
    for name in REQUIRED_DATASETS
}
PACKAGE_STATUS = {
    "status": "NOT_READY",
    "zip_authorized": False,
}

def write_status():
    write_json(
        ROOT / "ACQUISITION_STATUS.json",
        {
            "schema_version": 4,
            "updated_utc": utc_now(),
            "datasets": STATUSES,
            "package": PACKAGE_STATUS,
        },
    )

def attempt_dataset(name, fn):
    print("\n" + "=" * 80)
    print("ACQUIRING", name)
    print("=" * 80)
    STATUSES[name].update({
        "status": "RUNNING",
        "started_utc": utc_now(),
        "finished_utc": None,
        "detail": None,
    })
    write_status()
    try:
        result = fn()
        STATUSES[name].update({
            "status": "PASS",
            "finished_utc": utc_now(),
            "detail": result,
        })
        print(f"{name}: PASS")
    except Exception as exc:
        import traceback
        STATUSES[name].update({
            "status": "FAIL",
            "finished_utc": utc_now(),
            "detail": {
                "exception_type": type(exc).__name__,
                "message": str(exc),
                "traceback": traceback.format_exc(),
            },
        })
        print(f"{name}: FAIL — {type(exc).__name__}: {exc}")
    finally:
        write_status()

# The source lock is written before acquisition so a failed run still records
# the exact intended source identities and acceptance criteria.
write_json(
    ROOT / "SOURCES_LOCK.json",
    {
        "schema_version": 4,
        "created_utc": utc_now(),
        "operation_class": "acquisition_and_physical_byte_verification",
        "previous_acquisition_dependency": False,
        "mendeley_network_access": "NOT_USED",
        "sources": SOURCES,
    },
)


## BanglaBait acquisition

**Source URL:** `https://huggingface.co/datasets/Motahar/banglabait`  
**Source identity:** revision `ff22fb3b423952959a80dc5d9c6c3345b21cd145`  
**Required files:** `train.csv`, `dev.csv`, `test.csv`, `unlabelled_data.csv`  
**Frozen sizes:** 72,704,138; 7,969,172; 19,813,408; and 329,277,343 bytes respectively.  
**Expected SHA-256:** `train.csv`, `test.csv`, and `unlabelled_data.csv` have frozen SHA-256 values in the source lock. `dev.csv` is fixed by the pinned Git blob identity; its SHA-256 is independently calculated and recorded by this run.  
**Acquisition method:** direct Hugging Face `/resolve/<revision>/<file>` retrieval for the six pinned repository files. This avoids the high-request-count dataset metadata/snapshot API path that produced HTTP 429 in the prior run. The full immutable commit hash remains embedded in every download URL. A valid `HF_TOKEN` is used when present, but authentication does not alter the frozen revision or acceptance criteria.  
**Runtime dependencies:** `requests`, Python hashing and filesystem modules.  
**Network requirement:** six public Hugging Face resolver requests at the pinned revision. Resolver responses remain subject to retry-after handling.  
**Output artifact:** `EXTERNAL-DATASET-BYTES-v1/BanglaBait/`.  
**Failure criterion:** missing resolver object, frozen-size mismatch, frozen SHA-256 mismatch, or Git-blob mismatch. No mutable branch alias is used.


In [4]:
def acquire_banglabait():
    name = "BanglaBait"
    spec = SOURCES[name]
    stage, promoted = prepare_dataset_dir(name)
    snapshot_dir = stage / "snapshot"
    output_dir = stage / "promoted" / "repository_snapshot"
    snapshot_dir.mkdir(parents=True, exist_ok=True)

    token = os.environ.get("HF_TOKEN") or None
    auth_headers = {"Authorization": f"Bearer {token}"} if token else None

    verified = []
    retrievals = []

    # Direct resolver acquisition: one GET per immutable file, no dataset-info API.
    for rel, expected in spec["expected_tree"].items():
        encoded_rel = urllib.parse.quote(rel, safe="/")
        url = (
            f"https://huggingface.co/datasets/{spec['repo_id']}/resolve/"
            f"{spec['revision']}/{encoded_rel}?download=true"
        )
        path = snapshot_dir / rel
        path.parent.mkdir(parents=True, exist_ok=True)

        response = download_stream(
            url,
            path,
            timeout=(30, 1200),
            headers=auth_headers,
        )
        retrievals.append({
            "path": rel,
            "requested_url": url,
            "response": response,
            "authenticated": bool(token),
        })

        if path.stat().st_size != expected["size"]:
            raise RuntimeError(
                f"Size mismatch for {rel}: {path.stat().st_size} != {expected['size']}"
            )

        sha256 = sha256_file(path)
        record = {
            "path": rel,
            "size_bytes": path.stat().st_size,
            "sha256": sha256,
        }

        if expected.get("sha256"):
            if sha256 != expected["sha256"]:
                raise RuntimeError(
                    f"SHA-256 mismatch for {rel}: {sha256} != {expected['sha256']}"
                )
            record["frozen_sha256_verified"] = True
        else:
            blob = git_blob_sha1(path)
            if blob != expected["git_blob_sha1"]:
                raise RuntimeError(
                    f"Git blob mismatch for {rel}: {blob} != {expected['git_blob_sha1']}"
                )
            record["git_blob_sha1"] = blob
            record["git_blob_verified"] = True

        verified.append(record)

    actual_tree = {
        p.relative_to(snapshot_dir).as_posix()
        for p in snapshot_dir.rglob("*")
        if p.is_file()
    }
    expected_tree = set(spec["expected_tree"])
    if actual_tree != expected_tree:
        raise RuntimeError(
            f"BanglaBait tree mismatch: actual={sorted(actual_tree)} "
            f"expected={sorted(expected_tree)}"
        )

    shutil.copytree(snapshot_dir, output_dir)

    metadata = {
        "dataset": name,
        "source_url": spec["source_url"],
        "repo_id": spec["repo_id"],
        "requested_revision": spec["revision"],
        "resolved_identity_policy": (
            "Every physical file was retrieved through a URL containing the full "
            "locked commit hash; no branch alias or repository-metadata lookup was used."
        ),
        "retrieved_utc": utc_now(),
        "required_files": spec["required_files"],
        "verified_files": verified,
        "retrievals": retrievals,
        "hf_token_used": bool(token),
        "license_note": spec["license_note"],
    }
    write_json(stage / "promoted" / "SOURCE_METADATA.json", metadata)

    final_dir = promote_dataset(stage, promoted)
    return {
        "source_url": spec["source_url"],
        "revision": spec["revision"],
        "acquisition_method": "direct_pinned_huggingface_resolvers",
        "file_count": len(iter_files(final_dir)),
        "verified_files": verified,
    }

attempt_dataset("BanglaBait", acquire_banglabait)



ACQUIRING BanglaBait


.gitattributes:   0%|          | 0.00/2.61k [00:00<?, ?B/s]

README.md:   0%|          | 0.00/111 [00:00<?, ?B/s]

dev.csv:   0%|          | 0.00/7.97M [00:00<?, ?B/s]

test.csv:   0%|          | 0.00/19.8M [00:00<?, ?B/s]

train.csv:   0%|          | 0.00/72.7M [00:00<?, ?B/s]

unlabelled_data.csv:   0%|          | 0.00/329M [00:00<?, ?B/s]

BanglaBait: PASS



## BanglaClick acquisition

**Source URL:** `https://github.com/MBillahsust/BanglaClick`  
**Source identity:** commit `2ba28763d8f80eaba7a649877eae1fbbcb78a0e7`  
**Required files:** `Data/Final_4961_Dataset.xlsx`, `README.md`, `Data/README.md`  
**Expected file size:** the immutable Git tree at the pinned commit defines the expected blob sizes and is queried at runtime.  
**Expected SHA-256:** no independent pre-frozen SHA-256 was defined for these Git objects. The pinned commit and per-file Git blob SHA-1 define the expected physical bytes; SHA-256 is independently calculated and frozen in `SOURCE_METADATA.json` and the final acquisition manifest.  
**Acquisition method:** GitHub codeload archive for the exact commit, verified against the GitHub commit/tree API and local Git-blob reconstruction.  
**Runtime dependencies:** `requests`, `tarfile`, Python hashing modules.  
**Network requirement:** public GitHub API and codeload access.  
**Output artifact:** `EXTERNAL-DATASET-BYTES-v1/BanglaClick/`.  
**Failure criterion:** commit mismatch, required tree entry absent, blob size mismatch, local Git-blob mismatch, or archive-integrity failure.


In [5]:

def acquire_github_commit_dataset(name):
    spec = SOURCES[name]
    stage, promoted = prepare_dataset_dir(name)
    raw_dir = stage / "raw"
    extract_dir = stage / "extract"
    snapshot_out = stage / "promoted" / "repository_snapshot"
    raw_dir.mkdir(parents=True, exist_ok=True)
    extract_dir.mkdir(parents=True, exist_ok=True)

    commit_url = (
        f"https://api.github.com/repos/{spec['owner']}/{spec['repo']}"
        f"/commits/{spec['commit']}"
    )
    commit_meta = github_json(commit_url)
    resolved_commit = commit_meta.get("sha")
    if resolved_commit != spec["commit"]:
        raise RuntimeError(
            f"Commit mismatch: {resolved_commit} != {spec['commit']}"
        )

    tree_sha = (
        commit_meta.get("commit", {})
        .get("tree", {})
        .get("sha")
    )
    if not tree_sha:
        raise RuntimeError(
            f"GitHub commit metadata contains no tree SHA for {spec['commit']}"
        )

    tree_url = (
        f"https://api.github.com/repos/{spec['owner']}/{spec['repo']}"
        f"/git/trees/{tree_sha}?recursive=1"
    )
    tree_meta = github_json(tree_url)
    if tree_meta.get("sha") != tree_sha:
        raise RuntimeError(
            f"Git tree identity mismatch: {tree_meta.get('sha')} != {tree_sha}"
        )
    tree_entries = {
        item["path"]: item
        for item in tree_meta.get("tree", [])
        if item.get("type") == "blob"
    }

    missing = [p for p in spec["required_files"] if p not in tree_entries]
    if missing:
        raise RuntimeError(
            f"Required files absent from pinned Git tree: {missing}"
        )

    archive_url = (
        f"https://codeload.github.com/{spec['owner']}/{spec['repo']}"
        f"/tar.gz/{spec['commit']}"
    )
    archive_path = raw_dir / f"{spec['repo']}-{spec['commit']}.tar.gz"
    response = download_stream(archive_url, archive_path, timeout=(30, 300))
    archive_sha256 = sha256_file(archive_path)

    safe_extract_tar(archive_path, extract_dir)
    roots = [p for p in extract_dir.iterdir() if p.is_dir()]
    if len(roots) != 1:
        raise RuntimeError(
            f"Unexpected codeload archive structure: {[p.name for p in roots]}"
        )
    tree_root = roots[0]

    verified = []
    for rel in spec["required_files"]:
        local = tree_root / rel
        if not local.is_file():
            raise FileNotFoundError(local)

        remote = tree_entries[rel]
        remote_size = remote.get("size")
        if remote_size is not None and local.stat().st_size != int(remote_size):
            raise RuntimeError(
                f"Blob size mismatch for {rel}: "
                f"{local.stat().st_size} != {remote_size}"
            )

        local_blob = git_blob_sha1(local)
        remote_blob = remote.get("sha")
        if local_blob != remote_blob:
            raise RuntimeError(
                f"Git blob mismatch for {rel}: {local_blob} != {remote_blob}"
            )

        verified.append({
            "path": rel,
            "size_bytes": local.stat().st_size,
            "git_blob_sha1": local_blob,
            "sha256": sha256_file(local),
        })

    shutil.copytree(tree_root, snapshot_out)

    metadata = {
        "dataset": name,
        "source_url": spec["source_url"],
        "repo_url": spec["repo_url"],
        "requested_commit": spec["commit"],
        "resolved_commit": resolved_commit,
        "commit_api_url": commit_url,
        "tree_sha": tree_sha,
        "tree_api_url": tree_url,
        "archive_url": archive_url,
        "archive_response": response,
        "archive_size_bytes": archive_path.stat().st_size,
        "archive_sha256": archive_sha256,
        "required_files": spec["required_files"],
        "verified_files": verified,
        "sha256_policy": spec["sha256_policy"],
        "license_note": spec["license_note"],
        "retrieved_utc": utc_now(),
    }
    write_json(stage / "promoted" / "SOURCE_METADATA.json", metadata)

    final_dir = promote_dataset(stage, promoted)
    return {
        "source_url": spec["source_url"],
        "commit": resolved_commit,
        "file_count": len(iter_files(final_dir)),
        "verified_required_files": verified,
    }

attempt_dataset(
    "BanglaClick",
    lambda: acquire_github_commit_dataset("BanglaClick"),
)



ACQUIRING BanglaClick


BanglaClick-2ba28763d8f80eaba7a649877eae1fbbcb78a0e7.tar.gz: 0.00B [00:00, ?B/s]

BanglaClick: PASS



## BanMANI acquisition

**Source URL:** `https://github.com/kamruzzaman15/BanMANI`  
**Source identity:** commit `483be9ac31f550ee252d1df99d090ffdbcccc63f`  
**Required files:** `BanMANI.csv`, `README.md`, `LICENSE`  
**Expected file size:** the immutable Git tree at the pinned commit defines the expected blob sizes and is queried at runtime.  
**Expected SHA-256:** no independent pre-frozen SHA-256 was defined for these Git objects. The pinned commit and per-file Git blob SHA-1 define the expected physical bytes; SHA-256 is independently calculated and frozen in `SOURCE_METADATA.json` and the final acquisition manifest.  
**Acquisition method:** GitHub codeload archive for the exact commit, verified against the GitHub commit/tree API and local Git-blob reconstruction.  
**Runtime dependencies:** `requests`, `tarfile`, Python hashing modules.  
**Network requirement:** public GitHub API and codeload access.  
**Output artifact:** `EXTERNAL-DATASET-BYTES-v1/BanMANI/`.  
**Failure criterion:** commit mismatch, required tree entry absent, blob size mismatch, local Git-blob mismatch, or archive-integrity failure.


In [6]:

attempt_dataset(
    "BanMANI",
    lambda: acquire_github_commit_dataset("BanMANI"),
)



ACQUIRING BanMANI


BanMANI-483be9ac31f550ee252d1df99d090ffdbcccc63f.tar.gz: 0.00B [00:00, ?B/s]

BanMANI: PASS



## BanFakeNews-2.0 acquisition

**Authoritative release:** Mendeley Data Version 1, DOI `10.17632/kjh887ct4j.1`  
**Author source URL:** `https://www.kaggle.com/datasets/hrithikmajumdar/bangla-fake-news`  
**Pinned public fallback URL:** `https://huggingface.co/datasets/hrshihab/BanFakeNews-2.0` at commit `cef06ca5e2a90c408a28719d6d71c1e1388c3f7a`  
**Required main filename:** `All60Kdataset.csv`  
**Expected main size:** 304,586,219 bytes  
**Expected main SHA-256:** `d9aed69b6019843a769bbcb045affd31c98cfa02f2b0bc821f551c54ba780c0a`  
**Independent test source:** `https://github.com/Shibu4064/IndoNLP` at commit `8736467eeea8a441a35508777fcf6c8cdd323df7`  
**Required independent filename:** `New_Test_Dataset.csv`  
**Expected independent size:** 6,237,141 bytes  
**Expected independent SHA-256:** `ed4949762c1cdc1940bb5a1afe610ca1341a0803079888a35a812608f6351e7b`  
**Acquisition method:** authors' public Kaggle dataset is attempted first through the unauthenticated public dataset API; a pinned Hugging Face transport candidate is accepted only when its downloaded physical bytes exactly match the frozen Version 1 size and SHA-256. The independent test is downloaded from the pinned authors' GitHub commit.  
**Runtime dependencies:** `requests`, `huggingface_hub`, Python hashing modules.  
**Network requirement:** public Kaggle API, Hugging Face, and GitHub raw access. No Mendeley request is made.  
**Output artifact:** `EXTERNAL-DATASET-BYTES-v1/BanFakeNews-2.0/`.  
**Failure criterion:** failure of all public main-file routes, filename/size/SHA mismatch, or independent-test size/SHA mismatch.


In [7]:

def acquire_banfakenews():
    name = "BanFakeNews-2.0"
    spec = SOURCES[name]
    stage, promoted = prepare_dataset_dir(name)
    output_root = stage / "promoted"
    main_dir = output_root / "Mendeley_V1_byte_equivalent"
    test_dir = output_root / "Independent_Test_Authors_GitHub"
    main_dir.mkdir(parents=True, exist_ok=True)
    test_dir.mkdir(parents=True, exist_ok=True)

    main_name = "All60Kdataset.csv"
    main_lock = spec["files"][main_name]
    main_out = main_dir / main_name

    route_attempts = []
    accepted_route = None

    # Priority 1: authors' public Kaggle dataset.
    try:
        k = spec["author_kaggle"]
        route = kaggle_public_file_download(
            k["owner"],
            k["slug"],
            main_name,
            main_out,
            stage / "kaggle",
        )
        rec = verify_frozen_file(
            main_out,
            main_lock["size"],
            main_lock["sha256"],
        )
        accepted_route = {
            "route": route,
            "source_class": "author_owned_public_kaggle",
            "source_url": k["source_url"],
            "verification": rec,
        }
    except Exception as exc:
        route_attempts.append({
            "route": "author_owned_public_kaggle",
            "error": f"{type(exc).__name__}: {exc}",
        })
        if main_out.exists():
            bad = stage / "kaggle" / f"{main_name}.rejected"
            bad.parent.mkdir(parents=True, exist_ok=True)
            shutil.move(str(main_out), str(bad))

    # Priority 2: pinned public HF transport candidate.
    if accepted_route is None:
        try:
            hf = spec["public_hf_candidate"]
            cached = Path(hf_hub_download(
                repo_id=hf["repo_id"],
                repo_type="dataset",
                revision=hf["revision"],
                filename=main_name,
            ))
            cached_rec = verify_frozen_file(
                cached,
                main_lock["size"],
                main_lock["sha256"],
            )
            shutil.copyfile(cached, main_out)
            out_rec = verify_frozen_file(
                main_out,
                main_lock["size"],
                main_lock["sha256"],
            )
            accepted_route = {
                "route": {
                    "type": "pinned_public_huggingface_transport",
                    "repo_id": hf["repo_id"],
                    "revision": hf["revision"],
                    "source_url": hf["source_url"],
                    "cache_path": cached_rec["path"],
                },
                "source_class": "public_non_author_transport_hash_equivalent",
                "provenance_note": hf["provenance_note"],
                "verification": out_rec,
                "higher_priority_route_attempts": route_attempts,
            }
        except Exception as exc:
            route_attempts.append({
                "route": "pinned_public_huggingface_transport",
                "error": f"{type(exc).__name__}: {exc}",
            })

    if accepted_route is None:
        raise RuntimeError(
            "All60Kdataset.csv could not be acquired with the frozen Version 1 "
            f"byte identity. Attempts: {route_attempts}"
        )

    nt = spec["new_test"]
    nt_out = test_dir / nt["filename"]
    nt_response = download_stream(nt["raw_url"], nt_out, timeout=(30, 300))
    nt_rec = verify_frozen_file(nt_out, nt["size"], nt["sha256"])

    metadata = {
        "dataset": name,
        "authoritative_release": spec["authoritative"],
        "main_file": {
            "filename": main_name,
            "expected_size_bytes": main_lock["size"],
            "expected_sha256": main_lock["sha256"],
            "accepted_transport": accepted_route,
            "all_route_attempts": route_attempts,
        },
        "independent_test": {
            "repo_url": nt["repo_url"],
            "source_url": nt["source_url"],
            "commit": nt["commit"],
            "filename": nt["filename"],
            "raw_url": nt["raw_url"],
            "response": nt_response,
            "size_bytes": nt_rec["size_bytes"],
            "sha256": nt_rec["sha256"],
        },
        "retrieved_utc": utc_now(),
    }
    write_json(output_root / "SOURCE_METADATA.json", metadata)

    final_dir = promote_dataset(stage, promoted)
    return {
        "authoritative_release": spec["authoritative"],
        "main_file": accepted_route,
        "independent_test": {
            "size_bytes": nt_rec["size_bytes"],
            "sha256": nt_rec["sha256"],
            "commit": nt["commit"],
        },
        "file_count": len(iter_files(final_dir)),
    }

attempt_dataset("BanFakeNews-2.0", acquire_banfakenews)



ACQUIRING BanFakeNews-2.0


bangla-fake-news.zip:   0%|          | 0.00/67.6M [00:00<?, ?B/s]

All60Kdataset.csv:   0%|          | 0.00/305M [00:00<?, ?B/s]

New_Test_Dataset.csv:   0%|          | 0.00/6.24M [00:00<?, ?B/s]

BanFakeNews-2.0: PASS


## BaitBuster-Bangla acquisition

**Authoritative release identity:** Mendeley Data Version 3, DOI `10.17632/3c6ztw5nft.3`  
**Accessible public transport:** author Hugging Face repository `https://huggingface.co/datasets/abdalimran/BaitBuster-Bangla`  
**Pinned Hugging Face commit:** `be1528ba40dd8828c5c06ee70b7574cca22b8ee1`  

### Canonical research serialization

**Filename:** `BaitBuster-Bangla_253070_18c_HL10k_AIL.parquet`  
**Serialization role:** required canonical research artifact for downstream P0 analysis and experiments  
**Expected size:** 175,950,911 bytes  
**Expected SHA-256:** `7f1bb757c9c5ee1a2922efcd45fb829b89e96d4465995e538158c4225f3c12c9`  
**Acquisition method:** direct author Hugging Face `/resolve/<commit>/<file>` retrieval at the pinned commit, followed by independent physical size and SHA-256 verification.  
**Runtime dependencies:** `requests` and Python hashing/filesystem modules.  
**Network requirement:** public Hugging Face resolver access; `HF_TOKEN` is used when present without altering the pinned source identity.  
**Output artifact:** `EXTERNAL-DATASET-BYTES-v1/BaitBuster-Bangla/Mendeley_V3_public_author_serializations/BaitBuster-Bangla_253070_18c_HL10k_AIL.parquet`  
**Failure criterion:** missing object, byte-size mismatch, or SHA-256 mismatch.

### Secondary reference serialization

**Filename:** `BaitBuster-Bangla_253070_18c_HL10k_AIL.xlsx`  
**Serialization role:** secondary provenance/reference artifact; not required for BaitBuster `PASS`  
**Expected size:** 91,845,301 bytes  
**Expected SHA-256:** `a6d207d94ba0e4735a92478ce01c0080f79e9ce2d1557c78c304e5275b3d5286`  
**Acquisition method:** direct author Hugging Face resolver retrieval at the same pinned commit, followed by independent physical size and SHA-256 verification.  
**Failure handling:** failure is recorded in source metadata but does not invalidate the canonical Parquet artifact.

### Publisher-listed unavailable serialization

**Filename:** `BaitBuster-Bangla_253070_18c_HL10k_AIL.csv`  
**Publisher-listed target size:** 936,567,497 bytes  
**Publisher-listed target SHA-256:** `c94e1abd737fa53f983de2e3de2fc322757f5548024e463d8365c0e2a0e4ce63`  
**Public recovery status:** unavailable through the verified public author Hugging Face, author GitHub, and current author Kaggle sources investigated for this protocol.  
**Freeze role:** not required.  
**Reconstruction policy:** conversion from Parquet or XLSX is prohibited.

The Parquet and CSV serializations are not described as byte-equivalent. The canonical Parquet artifact is accepted solely as an author-owned public Version 3 artifact whose physical bytes independently match the locked Version 3 Parquet size and SHA-256.


In [8]:
def acquire_baitbuster():
    name = "BaitBuster-Bangla"
    spec = SOURCES[name]
    stage, promoted = prepare_dataset_dir(name)
    output_root = stage / "promoted"
    release_dir = output_root / "Mendeley_V3_public_author_serializations"
    release_dir.mkdir(parents=True, exist_ok=True)

    token = os.environ.get("HF_TOKEN") or None
    auth_headers = {"Authorization": f"Bearer {token}"} if token else None
    hf = spec["author_hf"]

    file_records = {}
    warnings = []

    def acquire_pinned_hf_artifact(artifact, role):
        filename = artifact["filename"]
        encoded = urllib.parse.quote(filename, safe="/")
        url = (
            f"https://huggingface.co/datasets/{hf['repo_id']}/resolve/"
            f"{hf['revision']}/{encoded}?download=true"
        )
        out = release_dir / filename
        response = download_stream(
            url,
            out,
            timeout=(30, 1200),
            headers=auth_headers,
        )
        rec = verify_frozen_file(
            out,
            artifact["size"],
            artifact["sha256"],
        )
        return {
            "role": role,
            "serialization": artifact["serialization"],
            "transport": "author_huggingface_direct_pinned_resolver",
            "source_url": hf["source_url"],
            "repo_id": hf["repo_id"],
            "revision": hf["revision"],
            "download_url": url,
            "response": response,
            "hf_token_used": bool(token),
            "size_bytes": rec["size_bytes"],
            "sha256": rec["sha256"],
            "physical_verification": "PASS",
        }

    # Required canonical research artifact.
    canonical = spec["canonical_research_artifact"]
    canonical_record = acquire_pinned_hf_artifact(
        canonical,
        role="canonical_research_serialization",
    )
    file_records[canonical["filename"]] = canonical_record

    # Secondary reference serialization. Failure is recorded but does not
    # invalidate the required canonical Parquet artifact.
    secondary = spec["secondary_reference_artifact"]
    try:
        secondary_record = acquire_pinned_hf_artifact(
            secondary,
            role="secondary_reference_serialization",
        )
        file_records[secondary["filename"]] = secondary_record
        secondary_status = "PASS"
    except Exception as exc:
        secondary_status = "UNAVAILABLE_OR_UNVERIFIED"
        warnings.append(
            f"{secondary['filename']}: {type(exc).__name__}: {exc}"
        )

    csv_status = {
        **spec["publisher_listed_unavailable_serialization"],
        "role": "publisher_listed_unavailable_serialization",
        "freeze_requirement": False,
        "byte_equivalence_to_parquet_claimed": False,
    }

    metadata = {
        "dataset": name,
        "authoritative_release": spec["authoritative"],
        "author_github": spec["author_github"],
        "accessible_public_transport": spec["author_hf"],
        "canonical_research_artifact": canonical_record,
        "secondary_reference_artifact": {
            "status": secondary_status,
            "record": file_records.get(secondary["filename"]),
        },
        "publisher_listed_csv": csv_status,
        "warnings": warnings,
        "license_note": spec["license_note"],
        "retrieved_utc": utc_now(),
        "pass_rule": (
            "BaitBuster-Bangla PASS requires exact physical verification of the "
            "pinned Version 3 Parquet canonical artifact. XLSX is secondary. "
            "The unavailable publisher-listed CSV is not required and is never reconstructed."
        ),
    }
    write_json(output_root / "SOURCE_METADATA.json", metadata)

    final_dir = promote_dataset(stage, promoted)
    return {
        "authoritative_release": spec["authoritative"],
        "canonical_research_artifact": canonical_record,
        "secondary_reference_artifact_status": secondary_status,
        "publisher_listed_csv_status": "DOCUMENTED_UNAVAILABLE_NOT_REQUIRED",
        "file_count": len(iter_files(final_dir)),
        "warnings": warnings,
    }

attempt_dataset("BaitBuster-Bangla", acquire_baitbuster)



ACQUIRING BaitBuster-Bangla


BaitBuster-Bangla_253070_18c_HL10k_AIL.parquet:   0%|          | 0.00/176M [00:00<?, ?B/s]

BaitBuster-Bangla_253070_18c_HL10k_AIL.xlsx:   0%|          | 0.00/91.8M [00:00<?, ?B/s]

BaitBuster-Bangla: PASS



## Immutable manifest construction and package completeness gate

All five acquisition attempts are completed before package authorization is evaluated. `SOURCES_LOCK.json`, `ACQUISITION_STATUS.json`, `ACQUISITION_MANIFEST.json`, `SHA256SUMS.txt`, and `README_HANDOFF.txt` are written independently of final ZIP authorization so that incomplete runs retain a complete diagnostic and provenance record.

`EXTERNAL-DATASET-BYTES-v1.zip` is created only when all five required datasets have physical `PASS` status.


In [9]:

all_pass = all(
    STATUSES[name]["status"] == "PASS"
    for name in REQUIRED_DATASETS
)

PACKAGE_STATUS.update({
    "status": "PASS" if all_pass else "FAIL",
    "zip_authorized": bool(all_pass),
    "updated_utc": utc_now(),
})
write_status()

dataset_manifest = {}
for name in REQUIRED_DATASETS:
    dataset_dir = ROOT / name
    files = iter_files(dataset_dir) if dataset_dir.is_dir() else []
    dataset_manifest[name] = {
        "status": STATUSES[name]["status"],
        "file_count": len(files),
        "total_bytes": sum(p.stat().st_size for p in files),
        "files": [
            {
                "path": p.relative_to(ROOT).as_posix(),
                "size_bytes": p.stat().st_size,
                "sha256": sha256_file(p),
            }
            for p in files
        ],
    }

write_json(
    ROOT / "ACQUISITION_MANIFEST.json",
    {
        "schema_version": 4,
        "freeze_id": PACKAGE_NAME,
        "created_utc": utc_now(),
        "operation_class": "acquisition_and_physical_byte_verification",
        "previous_acquisition_dependency": False,
        "all_required_pass": all_pass,
        "required_datasets": REQUIRED_DATASETS,
        "datasets": dataset_manifest,
        "status": STATUSES,
    },
)

readme = f"""Swarabyanjan external dataset immutable byte freeze
Freeze ID: {PACKAGE_NAME}
Created UTC: {utc_now()}

Operation class:
acquisition and physical-byte verification only

Previous acquisition dependency:
none

Mendeley credentials/network access:
not used

Source identities:
- BanglaBait: Motahar/banglabait at ff22fb3b423952959a80dc5d9c6c3345b21cd145
- BanglaClick: MBillahsust/BanglaClick at 2ba28763d8f80eaba7a649877eae1fbbcb78a0e7
- BanMANI: kamruzzaman15/BanMANI at 483be9ac31f550ee252d1df99d090ffdbcccc63f
- BanFakeNews-2.0: authoritative Mendeley V1 / DOI 10.17632/kjh887ct4j.1;
  public transport accepted only on exact frozen size and SHA-256 equality
- New_Test_Dataset.csv: Shibu4064/IndoNLP at
  8736467eeea8a441a35508777fcf6c8cdd323df7
- BaitBuster-Bangla: authoritative Mendeley V3 / DOI 10.17632/3c6ztw5nft.3;
  canonical research serialization = author-owned pinned V3 Parquet at
  be1528ba40dd8828c5c06ee70b7574cca22b8ee1;
  secondary reference serialization = author-owned pinned V3 XLSX;
  publisher-listed V3 CSV = documented unavailable and not required for the freeze

All required datasets PASS: {all_pass}

No preprocessing, deduplication, feature extraction, modeling, calibration,
statistical analysis, serialization conversion, or evaluation is performed in this notebook.

Original publisher/repository license terms remain controlling.
"""
(ROOT / "README_HANDOFF.txt").write_text(
    readme,
    encoding="utf-8",
    newline="\n",
)

# SHA256SUMS covers every promoted package file except itself.
sum_targets = [
    p for p in iter_files(ROOT)
    if p.name != "SHA256SUMS.txt"
]
(ROOT / "SHA256SUMS.txt").write_text(
    "\n".join(
        f"{sha256_file(p)}  {p.relative_to(ROOT).as_posix()}"
        for p in sum_targets
    ) + "\n",
    encoding="utf-8",
    newline="\n",
)

print("\n" + "=" * 80)
print("FINAL ACQUISITION STATUS")
print("=" * 80)
for name in REQUIRED_DATASETS:
    print(f"{name:22s} {STATUSES[name]['status']}")
    if STATUSES[name]["status"] != "PASS":
        detail = STATUSES[name].get("detail") or {}
        if isinstance(detail, dict):
            print("  ", detail.get("message"))
        else:
            print("  ", detail)
print(f"{'PACKAGE / ZIP':22s} {PACKAGE_STATUS['status']}")

if not all_pass:
    ZIP_PATH.unlink(missing_ok=True)
    ZIP_SHA_PATH.unlink(missing_ok=True)
    failed = [
        name for name in REQUIRED_DATASETS
        if STATUSES[name]["status"] != "PASS"
    ]
    print("\nFINAL ZIP: NOT CREATED")
    print("Failed datasets:", failed)
    print("Status record:", ROOT / "ACQUISITION_STATUS.json")
    print("Manifest:", ROOT / "ACQUISITION_MANIFEST.json")
    print("Staging diagnostics:", STAGING)
    raise RuntimeError(
        "Immutable external-dataset freeze incomplete; final ZIP prohibited. "
        f"Failed datasets: {failed}"
    )

# Deterministic member order and timestamps.
with zipfile.ZipFile(
    ZIP_PATH,
    "w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=6,
    allowZip64=True,
) as zf:
    for path in iter_files(ROOT):
        arcname = f"{PACKAGE_NAME}/{path.relative_to(ROOT).as_posix()}"
        info = zipfile.ZipInfo(
            arcname,
            date_time=(1980, 1, 1, 0, 0, 0),
        )
        info.compress_type = zipfile.ZIP_DEFLATED
        info.external_attr = 0o100644 << 16
        with open(path, "rb") as src, zf.open(
            info,
            "w",
            force_zip64=True,
        ) as dst:
            shutil.copyfileobj(
                src,
                dst,
                length=8 * 1024 * 1024,
            )

with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    bad_member = zf.testzip()
    zip_members = set(zf.namelist())

if bad_member is not None:
    ZIP_PATH.unlink(missing_ok=True)
    raise RuntimeError(
        f"Final ZIP CRC verification failed at {bad_member}"
    )

expected_members = {
    f"{PACKAGE_NAME}/{p.relative_to(ROOT).as_posix()}"
    for p in iter_files(ROOT)
}
if zip_members != expected_members:
    ZIP_PATH.unlink(missing_ok=True)
    raise RuntimeError(
        "Final ZIP member set differs from the verified package file set"
    )

zip_sha = sha256_file(ZIP_PATH)
ZIP_SHA_PATH.write_text(
    f"{zip_sha}  {ZIP_PATH.name}\n",
    encoding="utf-8",
    newline="\n",
)

print("\nFINAL FREEZE: PASS")
print("ZIP:", ZIP_PATH)
print("ZIP bytes:", f"{ZIP_PATH.stat().st_size:,}")
print("ZIP SHA-256:", zip_sha)
print("SHA sidecar:", ZIP_SHA_PATH)



FINAL ACQUISITION STATUS
BanglaBait             PASS
BanglaClick            PASS
BanMANI                PASS
BanFakeNews-2.0        PASS
BaitBuster-Bangla      PASS
PACKAGE / ZIP          PASS

FINAL FREEZE: PASS
ZIP: /kaggle/working/EXTERNAL-DATASET-BYTES-v1.zip
ZIP bytes: 373,309,863
ZIP SHA-256: e0de5c224f406fe22d8ad30804217465f789c49c70e1d3afbe00afd28deb7ccf
SHA sidecar: /kaggle/working/EXTERNAL-DATASET-BYTES-v1.zip.sha256


## Reproducibility handoff

The executed notebook, dataset-specific source metadata, source lock, acquisition status, acquisition manifest, SHA-256 manifest, handoff README, immutable ZIP, and ZIP SHA-256 sidecar constitute the acquisition-stage provenance record.

For downstream P0 auditing and experiments, the canonical BaitBuster input is:

`BaitBuster-Bangla/Mendeley_V3_public_author_serializations/BaitBuster-Bangla_253070_18c_HL10k_AIL.parquet`

The XLSX file, when successfully verified, is retained only as a secondary provenance/reference artifact. The unavailable publisher-listed CSV is neither required nor reconstructed.

Subsequent P0 auditing must consume the frozen dataset tree or ZIP read-only and must not modify the raw acquired bytes.
